# House Price Prediction - Machine Learning Project

This Jupyter Notebook presents a complete, end-to-end Machine Learning pipeline to predict house prices. It covers library imports, dataset generation/loading, data cleaning, preprocessing, exploratory data analysis, feature selection, model training, evaluation, and visualizations.

---

## Section 1: Import Libraries

In this section, we import all the required scientific computing, data manipulation, visualization, and machine learning libraries.

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import scikit-learn preprocessing and pipeline utilities
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Import scikit-learn models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Import scikit-learn metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("All libraries imported successfully!")

## Section 2: Load Dataset

We load the `house_price.csv` dataset. To make this notebook fully self-contained and executable in Google Colab/Jupyter out-of-the-box, we will automatically generate a realistic dataset if the file is not already present.

In [ ]:
import os

# Helper script to generate dataset if missing
if not os.path.exists("house_price.csv"):
    print("house_price.csv not found. Generating a realistic house price dataset...")
    np.random.seed(42)
    n_samples = 600
    
    sqft = np.random.normal(1800, 600, n_samples).astype(int)
    bedrooms = np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.1, 0.25, 0.45, 0.15, 0.05])
    bathrooms = np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0], size=n_samples)
    year_built = np.random.randint(1950, 2024, size=n_samples)
    garage_size = np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.15, 0.35, 0.4, 0.1])
    neighborhood = np.random.choice(['Downtown', 'Suburbs', 'Rural'], size=n_samples, p=[0.3, 0.5, 0.2])
    house_style = np.random.choice(['Modern', 'Classic', 'Ranch'], size=n_samples, p=[0.25, 0.45, 0.3])
    
    # Price modeling logic
    price = (sqft * 165) + (bedrooms * 18000) + (bathrooms * 12000) + ((year_built - 1950) * 900) + (garage_size * 15000)
    for i in range(n_samples):
        if neighborhood[i] == 'Downtown':
            price[i] += 60000
        elif neighborhood[i] == 'Suburbs':
            price[i] += 30000
            
    # Add random noise
    price += np.random.normal(0, 20000, n_samples)
    price = np.clip(price, 60000, None).astype(int)
    
    df_gen = pd.DataFrame({
        'square_feet': sqft,
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'year_built': year_built,
        'garage_size': garage_size,
        'neighborhood': neighborhood,
        'house_style': house_style,
        'price': price
    })
    
    # Inject missing values (NaNs)
    df_gen.loc[df_gen.sample(frac=0.06, random_state=12).index, 'square_feet'] = np.nan
    df_gen.loc[df_gen.sample(frac=0.04, random_state=34).index, 'neighborhood'] = np.nan
    
    # Inject duplicates
    df_gen = pd.concat([df_gen, df_gen.iloc[:20]], ignore_index=True)
    
    # Inject outliers
    df_gen.loc[df_gen.sample(n=8, random_state=56).index, 'price'] = df_gen['price'] * 3.8
    
    df_gen.to_csv("house_price.csv", index=False)
    print("Dataset created and saved to 'house_price.csv'!")

# Load the dataset
df = pd.read_csv("house_price.csv")

# Display the first 5 rows
print("--- First 5 Rows of the Dataset ---")
display(df.head())

# Display shape
print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Display column data types
print("--- Column Data Types ---")
print(df.dtypes)

# Display summary statistics
print("\n--- Summary Statistics (Numerical columns) ---")
display(df.describe())

# Display missing values count
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

## Section 3: Data Cleaning

Here we handle duplicate rows, missing numerical values using median, missing categorical values using the most frequent category, and remove outliers using the Interquartile Range (IQR) method.

In [ ]:
# Remove duplicates
initial_shape = df.shape
df.drop_duplicates(inplace=True)
print(f"Removed {initial_shape[0] - df.shape[0]} duplicate rows. New shape: {df.shape}")

In [ ]:
# Handle missing numerical values using Median
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled missing values in numerical column '{col}' with median: {median_val}")

# Handle missing categorical values using Most Frequent Value (Mode)
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled missing values in categorical column '{col}' with mode: '{mode_val}'")

In [ ]:
# Detect outliers using boxplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(ax=axes[0], y=df['price'], color='salmon')
axes[0].set_title('Price Distribution Boxplot (With Outliers)', fontsize=12)
axes[0].set_ylabel('Price ($)', fontsize=10)

sns.boxplot(ax=axes[1], y=df['square_feet'], color='skyblue')
axes[1].set_title('Square Feet Distribution Boxplot', fontsize=12)
axes[1].set_ylabel('Square Feet', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Remove extreme outliers using the IQR method on target variable 'price'
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_mask = (df['price'] < lower_bound) | (df['price'] > upper_bound)
print(f"IQR Bounds for price: {lower_bound} to {upper_bound}")
print(f"Number of outliers detected: {df[outliers_mask].shape[0]}")

# Filtering out outliers
df_clean = df[~outliers_mask].copy()
print(f"Shape after removing outliers: {df_clean.shape}")

In [ ]:
# Verify cleaned dataset contains no missing values
print("--- Missing values in cleaned dataset ---")
print(df_clean.isnull().sum())

print(f"\nVerification completed. Final shape: {df_clean.shape}")

## Section 4: Data Wrangling

We split the numerical and categorical columns, define preprocessing steps for each, and build a unified `ColumnTransformer` preprocessing pipeline.

In [ ]:
# Separate numerical and categorical columns (excluding target variable 'price')
target_col = 'price'
features = df_clean.drop(columns=[target_col])

numerical_cols = features.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")

In [ ]:
# Define Numerical transformer pipeline (Imputer + Scaler)
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Define Categorical transformer pipeline (Imputer + One-Hot Encoding)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Bundle transformers in a ColumnTransformer preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

print("Preprocessing pipeline configured successfully!")

## Section 5: Exploratory Data Analysis

Let's explore the relationships and structures of our variables through high-quality visualizations.

In [ ]:
# Histogram of target variable 'price'
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['price'], kde=True, color='purple', bins=30)
plt.title('Distribution of House Prices (Cleaned Target Variable)', fontsize=14, pad=15)
plt.xlabel('Price ($)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.show()

In [ ]:
# Correlation heatmap of numerical features
plt.figure(figsize=(8, 6))
corr_matrix = df_clean[numerical_cols + [target_col]].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, pad=15)
plt.show()

In [ ]:
# Pairplot of important numerical features
important_cols = ['price', 'square_feet', 'bedrooms', 'bathrooms']
sns.pairplot(df_clean[important_cols], diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot of Important Numerical Features', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Scatter plot between square_feet (important feature) and price
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x='square_feet', y='price', hue='neighborhood', alpha=0.7)
plt.title('Square Feet vs Price by Neighborhood', fontsize=14, pad=15)
plt.xlabel('Square Feet (sqft)', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(title='Neighborhood')
plt.show()

In [ ]:
# Boxplots for categorical features vs target variable price
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(ax=axes[0], data=df_clean, x='neighborhood', y='price', palette='Set2')
axes[0].set_title('Neighborhood vs Price', fontsize=13)
axes[0].set_xlabel('Neighborhood', fontsize=11)
axes[0].set_ylabel('Price ($)', fontsize=11)

sns.boxplot(ax=axes[1], data=df_clean, x='house_style', y='price', palette='Set3')
axes[1].set_title('House Style vs Price', fontsize=13)
axes[1].set_xlabel('House Style', fontsize=11)
axes[1].set_ylabel('Price ($)', fontsize=11)

plt.tight_layout()
plt.show()

## Section 6: Feature Selection

We evaluate feature relevance to select inputs for our model. We will analyze correlations and set our independent variable matrix `X` and dependent target vector `y`.

In [ ]:
# Show correlation of all numerical features with Target Variable (Price)
correlations = df_clean[numerical_cols + [target_col]].corr()[target_col].sort_values(ascending=False)
print("--- Correlation with Target Variable (Price) ---")
print(correlations)

# All features show significant correlations, so we keep all of them.
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print(f"\nFeature Matrix X shape: {X.shape}")
print(f"Target Vector y shape: {y.shape}")

## Section 7: Train-Test Split

We partition the dataset into 80% training set and 20% test set, using a fixed random state for reproducibility.

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## Section 8: Model Implementation

We train three models using Scikit-Learn pipelines to prevent data leakage during preprocessing:
1. **Linear Regression**
2. **Decision Tree Regressor**
3. **Random Forest Regressor**

In [ ]:
# Define the models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=6),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100, max_depth=8)
}

# Dictionary to store trained pipelines
trained_pipelines = {}

# Iterate and train pipelines
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline
    print(f"{name} Pipeline trained successfully!")

## Section 9: Model Evaluation

We compute regression performance metrics (Mean Absolute Error, Mean Squared Error, Root Mean Squared Error, and R² Score) for all models.

In [ ]:
evaluation_results = []

# Calculate evaluation metrics
for name, pipeline in trained_pipelines.items():
    preds = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    
    evaluation_results.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2
    })

# Build comparison DataFrame
results_df = pd.DataFrame(evaluation_results)
print("--- Model Evaluation Comparison Table ---")
display(results_df.sort_values(by='R2 Score', ascending=False))

## Section 10: Best Model Selection

We analyze model metrics to determine the best performing algorithm.

In [ ]:
# Find best model by R2 score
best_row = results_df.loc[results_df['R2 Score'].idxmax()]
best_model_name = best_row['Model']
best_r2 = best_row['R2 Score']

print(f"The Best Performing Model is: {best_model_name} with an R2 Score of {best_r2:.4f}")

### Why does it perform better?
1. **Linear Regression** assumes linear relationships and is susceptible to high bias if nonlinear relationships exist.
2. **Decision Trees** can model nonlinear interactions but suffer from high variance and overfitting if not carefully pruned.
3. **Random Forests** (Ensemble of Decision Trees) reduce model variance by bagging multiple randomized decision tree estimators, leading to smoother decision boundaries, better generalization, and robustness to outliers.

## Section 11: Prediction on a Sample House

We construct a sample house scenario and use our best model pipeline to predict its price.

In [ ]:
# Define unseen sample observation
sample_house = pd.DataFrame({
    'square_feet': [2100.0],
    'bedrooms': [3],
    'bathrooms': [2.0],
    'year_built': [2005],
    'garage_size': [2],
    'neighborhood': ['Suburbs'],
    'house_style': ['Modern']
})

print("--- Sample House Input Features ---")
display(sample_house)

# Extract and predict using the best model pipeline
best_pipeline = trained_pipelines[best_model_name]
predicted_price = best_pipeline.predict(sample_house)[0]

print(f"\nPredicted House Price using '{best_model_name}': ${predicted_price:,.2f}")

## Section 12: Visualization of Performance

We visualize our model predictions using four key regression validation plots: Actual vs Predicted, Residuals, Feature Importance, and Prediction Error.

In [ ]:
best_preds = best_pipeline.predict(X_test)
residuals = y_test - best_preds

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Actual vs Predicted Plot
sns.scatterplot(ax=axes[0, 0], x=y_test, y=best_preds, alpha=0.6, color='b')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_title(f'Actual vs Predicted House Prices ({best_model_name})', fontsize=12)
axes[0, 0].set_xlabel('Actual Price ($)', fontsize=10)
axes[0, 0].set_ylabel('Predicted Price ($)', fontsize=10)

# Plot 2: Residual Plot
sns.scatterplot(ax=axes[0, 1], x=best_preds, y=residuals, alpha=0.6, color='g')
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_title('Residuals vs Predicted Values', fontsize=12)
axes[0, 1].set_xlabel('Predicted Price ($)', fontsize=10)
axes[0, 1].set_ylabel('Residuals (Actual - Predicted)', fontsize=10)

# Plot 3: Feature Importance (Random Forest or Decision Tree)
if 'Random Forest' in trained_pipelines:
    rf_model = trained_pipelines['Random Forest'].named_steps['regressor']
    # Get one-hot encoder feature names
    ohe_cols = trained_pipelines['Random Forest'].named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols).tolist()
    all_features_names = numerical_cols + ohe_cols
    
    importances = rf_model.feature_importances_
    forest_importances = pd.Series(importances, index=all_features_names).sort_values(ascending=False)
    
    sns.barplot(ax=axes[1, 0], x=forest_importances.values, y=forest_importances.index, palette='viridis')
    axes[1, 0].set_title('Random Forest Feature Importances', fontsize=12)
    axes[1, 0].set_xlabel('Mean Decrease in Impurity', fontsize=10)
else:
    axes[1, 0].text(0.5, 0.5, 'Feature Importance Not Available', ha='center', va='center')

# Plot 4: Prediction Error Plot
sns.histplot(ax=axes[1, 1], data=residuals, kde=True, color='orange', bins=20)
axes[1, 1].set_title('Distribution of Prediction Errors (Residuals)', fontsize=12)
axes[1, 1].set_xlabel('Residuals ($)', fontsize=10)
axes[1, 1].set_ylabel('Count', fontsize=10)

plt.tight_layout()
plt.show()

## Section 13: Conclusion

### Dataset Overview:
The dataset consists of various numerical (square feet, bedrooms, bathrooms, year built, garage size) and categorical features (neighborhood, house style) representing physical features of residential houses, with the target column being `price`.

### Cleaning Performed:
- Duplicate rows were identified and removed.
- Missing numerical values in `square_feet` were filled using the feature median.
- Missing categorical values in `neighborhood` were filled using the feature mode.
- Extreme price outliers were detected using boxplots and removed using the IQR method (1.5 IQR threshold).

### Wrangling Steps:
- Independent features `X` and dependent target variable `y` were separated.
- A unified column preprocessing pipeline was created using Scikit-Learn `ColumnTransformer`:
  - Numerical columns were scaled using `StandardScaler` after imputer step.
  - Categorical columns were encoded using `OneHotEncoder` after imputer step.

### Best Model:
The **Random Forest Regressor** outperformed both Linear Regression and the Decision Tree Regressor by leveraging ensemble bagging to minimize prediction variance and capture complex interactions between variables.

### Final Conclusion:
Using preprocessing pipelines ensures clean, reproducible ML deployments while avoiding data leakage. Future improvements can include hyperparameter tuning via GridSearchCV/RandomizedSearchCV and testing gradient boosted tree architectures (e.g., XGBoost, LightGBM).